# Data Preparation

This notebook prepares the CrisisMMD dataset for practical experiments. It covers the creation of robust and reproducible data splits for unimodal text, unimodal image, and multimodal (text+image) classification tasks. The notebook also generates and saves text embeddings and adapts official CrisisMMD splits for benchmarking. Each step ensures class balance and prevents data leakage between splits, enabling fair model evaluation.

## Imports and Setup

This section imports all required libraries and loads environment variables and file paths. It also reads the unified, preprocessed annotations file into a DataFrame for further processing. Key parameters such as random seed and split proportions are defined here to ensure reproducibility throughout the data preparation

In [1]:
import numpy as np
import os
import pandas as pd

from sklearn.model_selection import train_test_split
from text_embeddings import generate_bert_embeddings, generate_bertweet_embeddings, generate_roberta_embeddings


False


In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
# Load paths from .env file
data_dir = os.getenv("DATA_DIR")
crisismmd_datasplits_dir = os.getenv("CRISISMMD_DATASPLITS_DIR")
datasplits_dir = os.getenv("DATASPLITS_DIR")
adapted_datasplits_dir = os.getenv("ADAPTED_DATASPLITS_DIR")
embeddings_dir = os.getenv("EMBEDDINGS_DIR")

# Load the unified preprocessed TSV file with image hashes
annotations_file = os.path.join(data_dir, "preprocessed_annotations.tsv")

In [4]:
df = pd.read_csv(annotations_file, sep='\t')

In [5]:
df.shape

(18082, 27)

In [6]:
df.head()

,tweet_id,image_id,text_info,text_info_conf,image_info,image_info_conf,text_human,text_human_conf,image_human,image_human_conf,...,img_width,img_height,img_aspect_ratio,img_hash_str,text_dup,image_dup,mm_info,mm_human,cleaned_text_bert,cleaned_text_bertweet
0,917791044158185473,917791044158185473_0,informative,1.0000,informative,0.6766,other_relevant_information,1.0000,other_relevant_information,0.6766,...,800,450,1.777778,86c476bb39c2b16c,False,False,informative,other_relevant_information,Wildfires raging through Northern California a...,Wildfires raging through Northern California a...
1,917791130590183424,917791130590183424_0,informative,1.0000,informative,0.6667,infrastructure_and_utility_damage,1.0000,affected_individuals,0.6667,...,1200,677,1.772526,f28e9aa98b96a730,False,False,informative,infrastructure_and_utility_damage,PHOTOS: Deadly wildfires rage in California,PHOTOS: Deadly wildfires rage in California ht...
2,917791291823591425,917791291823591425_0,informative,0.6813,informative,1.0000,other_relevant_information,0.6813,infrastructure_and_utility_damage,1.0000,...,640,480,1.333333,a9bdb18391838bba,False,False,informative,infrastructure_and_utility_damage,"PLS SHARE: We're capturing wildfire response, ...","PLS SHARE: We're capturing wildfire response, ..."
3,917791291823591425,917791291823591425_1,informative,0.6813,not_informative,1.0000,other_relevant_information,0.6813,not_humanitarian,1.0000,...,1200,900,1.333333,b582566fab45cac8,True,False,informative,other_relevant_information,"PLS SHARE: We're capturing wildfire response, ...","PLS SHARE: We're capturing wildfire response, ..."
4,917792092100988929,917792092100988929_0,informative,0.6727,informative,0.6612,other_relevant_information,0.6727,infrastructure_and_utility_damage,0.6612,...,600,400,1.500000,86826be7394ed43a,False,False,informative,infrastructure_and_utility_damage,California's raging wildfires as you've never ...,California's raging wildfires as you've never ...


In [7]:
# Parameters
RANDOM_SEED = 42
TRAIN_PERC = 0.70
VAL_PERC = 0.15
TEST_PERC = 0.15

## Unimodal Text Classification Data Split

**Goal:**  
Prepare data for text-only classification tasks by ensuring each tweet is represented only once and that class proportions are preserved across splits.

**Key Steps:**
1. **Remove Duplicates:** Exclude all entries where `text_dup` is `True` to ensure only unique tweets are used.
2. **Select Columns:** Keep only columns relevant for text classification: `tweet_text`, `text_info`, `text_human`, `cleaned_text_bert`, and `cleaned_text_bertweet`.
3. **Stratified Split:** Split the dataset into train (70%), validation (15%), and test (15%) sets using stratified sampling on the `text_human` label to maintain class balance.
4. **Save Splits:** Export each split as a TSV file for downstream modeling.

This ensures robust, reproducible splits for fair model evaluation and comparison.

In [8]:
# Remove text duplicates
text_df = df[df['text_dup'] == False].copy()

# Select relevant columns
cols = ['tweet_text', 'text_info', 'text_human', 'cleaned_text_bert', 'cleaned_text_bertweet']
text_df = text_df[cols]

In [9]:
text_df.shape

(16058, 5)

In [10]:
text_df.head()

,tweet_text,text_info,text_human,cleaned_text_bert,cleaned_text_bertweet
0,RT @Gizmodo: Wildfires raging through Northern...,informative,other_relevant_information,Wildfires raging through Northern California a...,Wildfires raging through Northern California a...
1,PHOTOS: Deadly wildfires rage in California ht...,informative,infrastructure_and_utility_damage,PHOTOS: Deadly wildfires rage in California,PHOTOS: Deadly wildfires rage in California ht...
2,RT @Cal_OES: PLS SHARE: We're capturing wildfi...,informative,other_relevant_information,"PLS SHARE: We're capturing wildfire response, ...","PLS SHARE: We're capturing wildfire response, ..."
4,RT @TIME: California's raging wildfires as you...,informative,other_relevant_information,California's raging wildfires as you've never ...,California's raging wildfires as you've never ...
5,Wildfires Threaten California's First Legal Ca...,informative,other_relevant_information,Wildfires Threaten California's First Legal Ca...,Wildfires Threaten California's First Legal Ca...


In [11]:
# Stratified split
# First split train vs temp (val+test)
text_df_train, text_df_temp = train_test_split(
    text_df,
    test_size=(1-TRAIN_PERC),
    stratify=text_df['text_human'],
    random_state=RANDOM_SEED
)

# Then split temp into val and test
val_size = VAL_PERC / (VAL_PERC + TEST_PERC)  # fraction of temp to go to val
text_df_val, text_df_test = train_test_split(
    text_df_temp,
    test_size=1-val_size,
    stratify=text_df_temp['text_human'],
    random_state=RANDOM_SEED
)

In [12]:
def print_class_distribution(df, label_col):
    """
    Print the class distribution for a given label column in a DataFrame.

    Args:
        df (pd.DataFrame): The DataFrame containing the data.
        label_col (str): The name of the column containing class labels.

    Prints:
        A DataFrame showing the count and percentage of each class.
    """
    # Count the occurrences of each class
    counts = df[label_col].value_counts(dropna=False)
    # Calculate the percentage of each class
    percentages = df[label_col].value_counts(normalize=True, dropna=False) * 100
    # Display the results as a DataFrame
    print(pd.DataFrame({'count': counts, 'percent': percentages.round(2)}))

In [13]:
# Disolay the sizes of the splits
print(f"Train: {len(text_df_train)}, Val: {len(text_df_val)}, Test: {len(text_df_test)}")

# Display class distributions for each split
print(f"\nTrain set class distribution:")
print_class_distribution(text_df_train, 'text_info')
print_class_distribution(text_df_train, 'text_human')
print(f"\nValidation set class distribution:")
print_class_distribution(text_df_val, 'text_info')
print_class_distribution(text_df_val, 'text_human')
print(f"\nTest set class distribution:")
print_class_distribution(text_df_test, 'text_info')
print_class_distribution(text_df_test, 'text_human')

Train: 11240, Val: 2409, Test: 2409

Train set class distribution:
                 count  percent
text_info                      
informative       8053    71.65
not_informative   3187    28.35
                                        count  percent
text_human                                            
other_relevant_information               4168    37.08
not_humanitarian                         3187    28.35
rescue_volunteering_or_donation_effort   2304    20.50
infrastructure_and_utility_damage         883     7.86
affected_individuals                      698     6.21

Validation set class distribution:
                 count  percent
text_info                      
informative       1726    71.65
not_informative    683    28.35
                                        count  percent
text_human                                            
other_relevant_information                893    37.07
not_humanitarian                          683    28.35
rescue_volunteering_or_donation_effo

In [14]:
# Save splits
train_save_path = os.path.join(datasplits_dir, "text_train.tsv")
val_save_path = os.path.join(datasplits_dir, "text_val.tsv")
test_save_path = os.path.join(datasplits_dir, "text_test.tsv")

text_df_train.to_csv(train_save_path, sep='\t', index=False)
text_df_val.to_csv(val_save_path, sep='\t', index=False)
text_df_test.to_csv(test_save_path, sep='\t', index=False)

## [Optional] Convert Text to Embeddings

**Goal:**  
Generate and store dense vector representations of tweets for use in neural models.

**Key Steps:**
1. **Load Splits:** Ensure the train, validation, and test splits are loaded.
2. **Generate Embeddings:** Use pretrained models (BERT, RoBERTa, BERTweet) to convert cleaned tweet texts into embeddings.
3. **Save Embeddings:** Store the resulting numpy arrays for efficient reuse in model training and evaluation.

This step accelerates experimentation by decoupling embedding computation from model training.

In [ ]:
if 'text_df_train' not in globals():
    text_df_train = pd.read_csv(train_save_path,sep = '\t')
    print(f"text train set size: {len(text_df_train)}")

if 'text_df_val' not in globals():
    text_df_val = pd.read_csv(val_save_path,sep = '\t')
    print(f"text val set size: {len(text_df_val)}")

if 'text_df_test' not in globals():
    text_df_test = pd.read_csv(test_save_path,sep = '\t')
    print(f"text test set size: {len(text_df_test)}")


In [ ]:
train_bert_vecs = generate_bert_embeddings(text_df_train, "cleaned_text_bert")
np.save(os.path.join(embeddings_dir, "train_bert.npy"), train_bert_vecs)

val_bert_vecs = generate_bert_embeddings(text_df_val, "cleaned_text_bert")
np.save(os.path.join(embeddings_dir, "val_bert.npy"), val_bert_vecs)

test_bert_vecs = generate_bert_embeddings(text_df_test, "cleaned_text_bert")
np.save(os.path.join(embeddings_dir, "test_bert.npy"), test_bert_vecs)

print(f"{len(train_bert_vecs) + len(val_bert_vecs) + len(test_bert_vecs)} embeddings created and saved")
print(f"length of each embedding: {len(train_bert_vecs[0])}")

In [ ]:
train_roberta_vecs = generate_roberta_embeddings(text_df_train, "cleaned_text_bert")
np.save(os.path.join(embeddings_dir, "train_roberta.npy"), train_roberta_vecs)

val_roberta_vecs = generate_roberta_embeddings(text_df_val, "cleaned_text_bert")
np.save(os.path.join(embeddings_dir, "val_roberta.npy"), val_roberta_vecs)

test_roberta_vecs = generate_roberta_embeddings(text_df_test, "cleaned_text_bert")
np.save(os.path.join(embeddings_dir, "test_roberta.npy"), test_roberta_vecs)

print(f"{len(train_roberta_vecs) + len(val_roberta_vecs) + len(test_roberta_vecs)} embeddings created and saved")
print(f"length of each embedding: {len(train_roberta_vecs[0])}")

In [ ]:
train_bertweet_vecs = generate_bertweet_embeddings(text_df_train, "cleaned_text_bertweet")
np.save(os.path.join(embeddings_dir, "train_bertweet.npy"), train_bertweet_vecs)

val_bertweet_vecs = generate_bertweet_embeddings(text_df_val, "cleaned_text_bertweet")
np.save(os.path.join(embeddings_dir, "val_bertweet.npy"), val_bertweet_vecs)

test_bertweet_vecs = generate_bertweet_embeddings(text_df_test, "cleaned_text_bertweet")
np.save(os.path.join(embeddings_dir, "test_bertweet.npy"), test_bertweet_vecs)

print(f"{len(train_bertweet_vecs) + len(val_bertweet_vecs) + len(test_bertweet_vecs)} embeddings created and saved")
print(f"length of each embedding: {len(train_bertweet_vecs[0])}")

## Unimodal Image Classification Data Split

**Goal:**  
Prepare data for image-only classification tasks, ensuring each unique image is used only once and class proportions are preserved.

**Key Steps:**
1. **Remove Duplicates:** Exclude all entries where `image_dup` is `True` to ensure only unique images are included.
2. **Select Columns:** Retain only `img_hash_str`, `image_path`, `image_info`, and `image_human` for image-based modeling.
3. **Stratified Split:** Divide the data into train, validation, and test sets (70/15/15) using stratified sampling on the `image_human` label.
4. **Save Splits:** Write each split to a TSV file for use in image classification pipelines.

This process ensures that no image is seen in more than one split and that class distributions are consistent.

In [15]:
# Remove duplicated images
img_df = df[df['image_dup'] == False].copy()

# Select relevant columns
cols = ['img_hash_str', 'image_path', 'image_info', 'image_human']
img_df = img_df[cols]

In [16]:
img_df.shape

(17354, 4)

In [17]:
img_df.head()

,img_hash_str,image_path,image_info,image_human
0,86c476bb39c2b16c,data_image/california_wildfires/10_10_2017/917...,informative,other_relevant_information
1,f28e9aa98b96a730,data_image/california_wildfires/10_10_2017/917...,informative,affected_individuals
2,a9bdb18391838bba,data_image/california_wildfires/10_10_2017/917...,informative,infrastructure_and_utility_damage
3,b582566fab45cac8,data_image/california_wildfires/10_10_2017/917...,not_informative,not_humanitarian
4,86826be7394ed43a,data_image/california_wildfires/10_10_2017/917...,informative,infrastructure_and_utility_damage


In [18]:
# Douvle-check successful removal of duplicates
unique_img_hashes = img_df['img_hash_str'].nunique()
print(f"Number of unique img_hash_str values: {unique_img_hashes}")

Number of unique img_hash_str values: 17354


In [19]:
# Stratified split
# First split train vs temp (val+test)
img_df_train, img_df_temp = train_test_split(
    img_df,
    test_size=(1-TRAIN_PERC),
    stratify=img_df['image_human'],
    random_state=RANDOM_SEED
)

# Then split temp into val and test
val_size = VAL_PERC / (VAL_PERC + TEST_PERC)  # fraction of temp to go to val
img_df_val, img_df_test = train_test_split(
    img_df_temp,
    test_size=1-val_size,
    stratify=img_df_temp['image_human'],
    random_state=RANDOM_SEED
)

In [20]:
# Display the sizes of the splits
print(f"Train: {len(img_df_train)}, Val: {len(img_df_val)}, Test: {len(img_df_test)}")

# Display class distributions for each split
print(f"\nTrain set class distribution:")
print_class_distribution(img_df_train, 'image_info')
print_class_distribution(img_df_train, 'image_human')
print(f"\nValidation set class distribution:")
print_class_distribution(img_df_val, 'image_info')
print_class_distribution(img_df_val, 'image_human')
print(f"\nTest set class distribution:")
print_class_distribution(img_df_test, 'image_info')
print_class_distribution(img_df_test, 'image_human')

Train: 12147, Val: 2603, Test: 2604

Train set class distribution:
                 count  percent
image_info                     
informative       6239    51.36
not_informative   5908    48.64
                                        count  percent
image_human                                           
not_humanitarian                         5908    48.64
infrastructure_and_utility_damage        2588    21.31
other_relevant_information               1690    13.91
rescue_volunteering_or_donation_effort   1509    12.42
affected_individuals                      452     3.72

Validation set class distribution:
                 count  percent
image_info                     
informative       1337    51.36
not_informative   1266    48.64
                                        count  percent
image_human                                           
not_humanitarian                         1266    48.64
infrastructure_and_utility_damage         555    21.32
other_relevant_information          

In [21]:
# Save splits
train_save_path = os.path.join(datasplits_dir, "img_train.tsv")
val_save_path = os.path.join(datasplits_dir, "img_val.tsv")
test_save_path = os.path.join(datasplits_dir, "img_test.tsv")

img_df_train.to_csv(train_save_path, sep='\t', index=False)
img_df_val.to_csv(val_save_path, sep='\t', index=False)
img_df_test.to_csv(test_save_path, sep='\t', index=False)

## Multimodal Classification Data Split

**Goal:**  
Create splits for multimodal (text+image) classification, ensuring each sample is unique and labels are consistent.

**Key Steps:**
1. **Select Columns:** Use only the columns needed for multimodal modeling: `cleaned_text_bert`, `image_path`, `mm_info`, `mm_human`, `text_dup`, `image_dup`.
2. **Remove Conflicts:** Exclude samples where `mm_human` is `"conflict"` to avoid ambiguous supervision.
3. **Remove Duplicates:** Keep only samples where both `text_dup` and `image_dup` are `False` to ensure uniqueness.
4. **Stratified Split:** Split into train, validation, and test sets (70/15/15), stratified by `mm_human` for balanced class representation.
5. **Clean Columns:** Drop unnecessary columns and rename for clarity.
6. **Save Splits:** Export the splits for use in multimodal experiments.

This ensures clean, non-overlapping, and balanced splits for multimodal learning.

In [22]:
# Select only relevant columns
columns = ["cleaned_text_bert", "image_path", "mm_info", "mm_human", "text_dup", "image_dup"]
mm_df = df[columns].copy()
mm_df.head(1)

,cleaned_text_bert,image_path,mm_info,mm_human,text_dup,image_dup
0,Wildfires raging through Northern California a...,data_image/california_wildfires/10_10_2017/917...,informative,other_relevant_information,False,False


In [23]:
# Remove conflicting labels for humanitarian labels
mm_df = mm_df[mm_df["mm_human"] != "conflict"]
mm_df.shape

(17715, 6)

In [24]:
# Remove both text and image duplicates
mm_df = mm_df[(mm_df["text_dup"] == False) & (mm_df["image_dup"] == False)]
mm_df.shape

(15070, 6)

In [25]:
# Stratified split
# First split train vs temp (val+test)
mm_train_df, mm_temp_df = train_test_split(
    mm_df,
    test_size=(1-TRAIN_PERC),
    stratify=mm_df["mm_human"],
    random_state=RANDOM_SEED
)

# Then split temp into val and test
val_size = VAL_PERC / (VAL_PERC + TEST_PERC)  # fraction of temp to go to val
mm_val_df, mm_test_df = train_test_split(
    mm_temp_df,
    test_size=(1-val_size),
    stratify=mm_temp_df["mm_human"],
    random_state=RANDOM_SEED
)

In [26]:
for split in [mm_train_df, mm_val_df, mm_test_df]:
    split.drop(columns=["text_dup", "image_dup"], inplace=True)  # Remove now unnecessary columns
    split.rename(columns={"cleaned_text_bert": "cleaned_tweet_text"}, inplace=True)  # Rename for semplicity

In [27]:
# Display the sizes of the splits
print(f"Train: {len(mm_train_df)}, Val: {len(mm_val_df)}, Test: {len(mm_test_df)}")

# Display class distributions for each split
print(f"\nTrain set class distribution:")
print_class_distribution(mm_train_df, 'mm_info')
print_class_distribution(mm_train_df, 'mm_human')
print(f"\nValidation set class distribution:")
print_class_distribution(mm_val_df, 'mm_info')
print_class_distribution(mm_val_df, 'mm_human')
print(f"\nTest set class distribution:")
print_class_distribution(mm_test_df, 'mm_info')
print_class_distribution(mm_test_df, 'mm_human')

Train: 10548, Val: 2261, Test: 2261

Train set class distribution:
                 count  percent
mm_info                        
informative       8004    75.88
not_informative   2544    24.12
                                        count  percent
mm_human                                              
not_humanitarian                         2977    28.22
other_relevant_information               2399    22.74
rescue_volunteering_or_donation_effort   2374    22.51
infrastructure_and_utility_damage        2181    20.68
affected_individuals                      617     5.85

Validation set class distribution:
                 count  percent
mm_info                        
informative       1712    75.72
not_informative    549    24.28
                                        count  percent
mm_human                                              
not_humanitarian                          638    28.22
other_relevant_information                515    22.78
rescue_volunteering_or_donation_effo

In [29]:
# Save splits
mm_train_df.to_csv(os.path.join(datasplits_dir, "mm_train.tsv"), sep="\t", index=False)
mm_val_df.to_csv(os.path.join(datasplits_dir, "mm_val.tsv"), sep="\t", index=False)
mm_test_df.to_csv(os.path.join(datasplits_dir, "mm_test.tsv"), sep="\t", index=False)

## [Optional] CrisisMMD Data Split Adaptation

**Goal:**  
Adapt the official CrisisMMD splits to include our cleaned text and image features for benchmarking and reproducibility.

**Key Steps:**
1. **Load Official Splits:** Import the original CrisisMMD train, validation, and test splits for both informative and humanitarian tasks.
2. **Merge Cleaned Features:** Join our cleaned text embeddings with the official splits using `tweet_id`.
3. **Select Columns:** For text, keep `tweet_text`, `label_text`, `cleaned_text_bert`, and `cleaned_text_bertweet`. For images, keep `image` and `label_image`.
4. **Save Adapted Splits:** Write the adapted splits to disk for direct use in benchmarking.

This enables direct comparison with published results while leveraging improved preprocessing and embeddings.

In [ ]:
# Load the premade datasplits that comes with the CrisisMMD dataset 
task1_train_df = pd.read_csv(os.path.join(crisismmd_datasplits_dir, "task_informative_text_img_agreed_lab_train.tsv"), sep='\t')
task1_val_df = pd.read_csv(os.path.join(crisismmd_datasplits_dir, "task_informative_text_img_agreed_lab_dev.tsv"), sep='\t')
task1_test_df = pd.read_csv(os.path.join(crisismmd_datasplits_dir, "task_informative_text_img_agreed_lab_test.tsv"), sep='\t')

task2_train_df = pd.read_csv(os.path.join(crisismmd_datasplits_dir, "task_humanitarian_text_img_agreed_lab_train.tsv"), sep='\t')
task2_val_df = pd.read_csv(os.path.join(crisismmd_datasplits_dir, "task_humanitarian_text_img_agreed_lab_dev.tsv"), sep='\t')
task2_test_df = pd.read_csv(os.path.join(crisismmd_datasplits_dir, "task_humanitarian_text_img_agreed_lab_test.tsv"), sep='\t')

In [ ]:
task1_train_df.head(1)

In [ ]:
task1_train_df.shape

In [ ]:
task1_val_df.shape

In [ ]:
task1_test_df.shape

In [ ]:
task2_train_df.head(1)

In [ ]:
task2_train_df.shape

In [ ]:
task2_val_df.shape

In [ ]:
task2_test_df.shape

In [ ]:
# Select relevant columns from CrisisMMD splits
text_columns_task = ["tweet_id", "tweet_text", "label_text"]
# Select columns we need to add to the CrisisMMD splits from our dataframe
text_embeddings = df[["tweet_id", "cleaned_text_bert", "cleaned_text_bertweet"]].drop_duplicates("tweet_id")

# Prepare text splits for CrisisMMD tasks by merging with cleaned text embeddings
# For each split, keep only unique tweet_texts and join with embeddings on tweet_id
task1_text_train_df = task1_train_df[text_columns_task].copy().drop_duplicates("tweet_text").merge(text_embeddings, on="tweet_id", how="left")[["tweet_text", "label_text", "cleaned_text_bert", "cleaned_text_bertweet"]]
task1_text_val_df = task1_val_df[text_columns_task].copy().drop_duplicates("tweet_text").merge(text_embeddings, on="tweet_id", how="left")[["tweet_text", "label_text", "cleaned_text_bert", "cleaned_text_bertweet"]]
task1_text_test_df = task1_test_df[text_columns_task].copy().drop_duplicates("tweet_text").merge(text_embeddings, on="tweet_id", how="left")[["tweet_text", "label_text", "cleaned_text_bert", "cleaned_text_bertweet"]]

task2_text_train_df = task2_train_df[text_columns_task].copy().drop_duplicates("tweet_text").merge(text_embeddings, on="tweet_id", how="left")[["tweet_text", "label_text", "cleaned_text_bert", "cleaned_text_bertweet"]]
task2_text_val_df = task2_val_df[text_columns_task].copy().drop_duplicates("tweet_text").merge(text_embeddings, on="tweet_id", how="left")[["tweet_text", "label_text", "cleaned_text_bert", "cleaned_text_bertweet"]]
task2_text_test_df = task2_test_df[text_columns_task].copy().drop_duplicates("tweet_text").merge(text_embeddings, on="tweet_id", how="left")[["tweet_text", "label_text", "cleaned_text_bert", "cleaned_text_bertweet"]]

In [ ]:
# For image splits, adaptation is easier since we already have the image paths and labels
# Select relevant columns from CrisisMMD splits
img_columns_task = ["image", "label_image"]

# Keep only image paths and labels
task1_img_train_df = task1_train_df[img_columns_task].copy()
task1_img_val_df = task1_val_df[img_columns_task].copy()
task1_img_test_df = task1_test_df[img_columns_task].copy()

task2_img_train_df = task2_train_df[img_columns_task].copy()
task2_img_val_df = task2_val_df[img_columns_task].copy()
task2_img_test_df = task2_test_df[img_columns_task].copy()

In [ ]:
# Create the directory if it doesn't exist
os.makedirs(adapted_datasplits_dir, exist_ok=True)

# Save splits
task1_text_train_df.to_csv(os.path.join(adapted_datasplits_dir, "task1_text_train.tsv"), sep='\t', index=False)
task1_text_val_df.to_csv(os.path.join(adapted_datasplits_dir, "task1_text_val.tsv"), sep='\t', index=False)
task1_text_test_df.to_csv(os.path.join(adapted_datasplits_dir, "task1_text_test.tsv"), sep='\t', index=False)
task1_img_train_df.to_csv(os.path.join(adapted_datasplits_dir, "task1_img_train.tsv"), sep='\t', index=False)
task1_img_val_df.to_csv(os.path.join(adapted_datasplits_dir, "task1_img_val.tsv"), sep='\t', index=False)
task1_img_test_df.to_csv(os.path.join(adapted_datasplits_dir, "task1_img_test.tsv"), sep='\t', index=False)
task2_text_train_df.to_csv(os.path.join(adapted_datasplits_dir, "task2_text_train.tsv"), sep='\t', index=False)
task2_text_val_df.to_csv(os.path.join(adapted_datasplits_dir, "task2_text_val.tsv"), sep='\t', index=False)
task2_text_test_df.to_csv(os.path.join(adapted_datasplits_dir, "task2_text_test.tsv"), sep='\t', index=False)
task2_img_train_df.to_csv(os.path.join(adapted_datasplits_dir, "task2_img_train.tsv"), sep='\t', index=False)
task2_img_val_df.to_csv(os.path.join(adapted_datasplits_dir, "task2_img_val.tsv"), sep='\t', index=False)
task2_img_test_df.to_csv(os.path.join(adapted_datasplits_dir, "task2_img_test.tsv"), sep='\t', index=False)